# Bản chú giải: Truy hồi điều luật tiếng Việt cho câu hỏi gõ không dấu

Phục hồi dấu, truy hồi lai và kiểm toán rò rỉ dữ liệu.
Nhóm 09, môn Xử lý ngôn ngữ tự nhiên. GVHD: Đặng Văn Thìn.

Notebook này chạy **cùng một mã** với `demo.ipynb`, chỉ dày thêm phần giải thích.
Bản kia để chiếu khi bảo vệ, bản này để đọc hiểu.

Mỗi mục dưới đây trả lời ba câu: *đang làm gì*, *vì sao làm thế*, và *nếu làm khác
thì hỏng ở đâu*.

Hai câu hỏi nghiên cứu:

- **CH1.** Trên dữ liệu đã kiểm toán, với mô hình ngữ nghĩa chưa thấy dữ liệu, ghép
  thêm BM25 còn cải thiện được bao nhiêu? Mục 7 trả lời.
- **CH2.** Khi người dùng gõ không dấu, hệ còn đứng vững không, và cách sửa rẻ nhất
  là gì? Mục 8 trả lời.

### Nội dung

1. Môi trường
2. Dữ liệu sau kiểm toán
3. Bằng chứng rò rỉ
4. Vì sao phải chia đoạn
5. Ba tầng trên một câu hỏi thật
6. Giải thích: từ nào đã khớp
7. Kết quả trên câu có dấu
8. Câu hỏi gõ không dấu
9. Hệ sai ở đâu
10. Trợ lý tra cứu

## 1. Môi trường

Máy có nhiều bản Python cùng đánh số 3.14.4 nhưng chỉ một bản đủ thư viện, nên
notebook phải chạy bằng kernel trỏ đúng vào `C:\Python314\python.exe`. Ô dưới in
ra đường dẫn thật để kiểm.

Card RTX 5060 Ti là kiến trúc Blackwell, cần wheel `cu130`. Nếu thấy hậu tố `+cpu`
thì mọi phần mã hóa sẽ chạy trên CPU và chậm hàng chục lần.

In [1]:
import json, sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import pandas as pd
import torch

from vlr import (chunking, config, dense, diacritics, explain, fusion, lexical,
                 metrics, pipeline, textnorm, tra_cuu)

pd.set_option("display.max_colwidth", 68)
pd.set_option("display.width", 150)
DOC = {"encoding": "utf-8-sig"}   # CSV ghi kèm BOM để mở bằng Excel không vỡ chữ

print("Python     :", sys.version.split()[0], "|", sys.executable)
print("torch      :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0),
          f"| VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("Thư mục gốc:", ROOT)

Python     : 3.14.4 | C:\Python314\python.exe
torch      : 2.12.0+cu130 | CUDA: True
GPU        : NVIDIA GeForce RTX 5060 Ti | VRAM 17.1 GB
Thư mục gốc: D:\03_Learning\Dai hoc_Thac si\VB2_AI_DHQG_HCM\hk3\laptrinhpythonchomayhoc\doanxulyngonngutn


## 2. Dữ liệu sau kiểm toán

Bộ gốc chỉ có `train` và `test`, **không có validation**. Đó là một cái bẫy: không
có val thì mọi tham số phải quét trên test, và con số công bố chính là con số đã
chỉnh cho vừa test.

Nhóm cắt val ra từ train theo **câu hỏi** chứ không theo cặp nhãn, `seed = 42`.
Chia theo cặp nhãn sẽ làm một câu hỏi có hai điều gold bị tách qua hai tập, và
cùng một câu được chấm ở cả hai nơi.

Chú ý cột `so_cap_qrel` khác `so_cau_hoi`: vài câu hỏi có nhiều hơn một điều luật
đúng.

In [2]:
qr = pd.read_parquet(config.QRELS_PATH)
qs = pd.read_parquet(config.QUERIES_PATH)
arts = pd.read_parquet(config.ARTICLES_PATH, columns=["article_id", "title", "text"])

print(f"{len(arts):,} điều luật  |  {len(qs):,} câu hỏi  |  {len(qr):,} cặp nhãn"
      .replace(",", "."))
display(pd.read_csv(config.AUDIT_DIR / "split_distribution.csv", **DOC))

61.425 điều luật  |  3.196 câu hỏi  |  3.271 cặp nhãn


,split,so_cau_hoi,so_cap_qrel,gold_trung_binh
0,test,788,793,1.006
1,train,1925,1986,1.032
2,val,481,492,1.023


## 3. Bằng chứng rò rỉ

Đây là phần quan trọng nhất của cả đồ án.

24 câu hỏi xuất hiện ở **cả** tệp train lẫn tệp test. Nếu chỉ dừng ở đó thì đơn
giản: loại khỏi train là xong. Nhưng khi đọc kỹ thì thấy chúng **không phải dòng
lặp**: với cả 24 trên 24 câu, tệp train chỉ sang một điều luật, tệp test chỉ sang
điều luật khác hẳn.

Cả hai đều là điều luật có thật và đều liên quan tới câu hỏi. Nghĩa là bộ nhãn
**không đầy đủ**, và mọi con số Recall trong báo cáo này là **cận dưới**: hệ có
thể trả về một điều luật đúng mà vẫn bị chấm sai chỉ vì người gán nhãn không liệt
kê điều đó.

Ô dưới còn chạy lại phép kiểm tự động: ba tập không được giao nhau câu hỏi nào.
Phép kiểm này nằm trong `scripts/01_prepare_data.py` dưới dạng `assert`, nên nó
chạy mỗi lần chuẩn bị dữ liệu chứ không chỉ ở đây.

In [3]:
xung = pd.read_csv(config.AUDIT_DIR / "conflicting_gold.csv", **DOC)
khac = int((xung["giong_nhau"].astype(str).str.lower() == "false").sum())
print(f"Câu hỏi nằm ở cả train và test: {len(xung)}")
print(f"Trong đó gold ở train KHÁC gold ở test: {khac} / {len(xung)}")
display(xung[["text", "gold_train", "gold_test"]].head(3))

ids = {s: set(g["query_id"]) for s, g in qr.groupby("split")}
print("Kiểm tra tự động, ba tập phải rời nhau:")
print(f"   train ∩ val  = {len(ids['train'] & ids['val'])}")
print(f"   train ∩ test = {len(ids['train'] & ids['test'])}")
print(f"   val   ∩ test = {len(ids['val'] & ids['test'])}")

Câu hỏi nằm ở cả train và test: 24
Trong đó gold ở train KHÁC gold ở test: 24 / 24


,text,gold_train,gold_test
0,"Bị thương tật 25%, cơ quan nhà nước có khởi tố không?",12/2017/qh14+1,101/2015/qh13+155
1,Tiêu chí xác định thương nhân áp dụng chế độ Luồng Đỏ được quy đ...,31/2018/nđ-cp+29,15/2018/tt-bct+6
2,Đi trường giáo dưỡng được gọi điện thoại về nhà 5 phút/lần được ...,43/2015/tt-bca+2,43/2015/tt-bca+12


Kiểm tra tự động, ba tập phải rời nhau:
   train ∩ val  = 0
   train ∩ test = 0
   val   ∩ test = 0


## 4. Vì sao phải chia đoạn

PhoBERT nhận tối đa 256 token, khoảng 140 từ tiếng Việt. **61,1% điều luật dài hơn
thế.** Cắt cụt là vứt phần đuôi, mà đuôi điều luật thường là chỗ ghi mức phạt và
các trường hợp ngoại lệ, đúng thứ người dân hay hỏi.

Cách cắt của nhóm: theo **ranh giới khoản** trước, chỉ khi một khoản tự nó dài hơn
180 từ mới cắt bằng cửa sổ trượt chồng lấn 40 từ. Cắt giữa câu làm hỏng nghĩa;
ranh giới khoản là ranh giới ngữ nghĩa có sẵn của văn bản luật.

Mỗi đoạn đều được ghép **tiêu đề điều** ở đầu. Đoạn thứ ba của một điều mà mất
tiêu đề thì gần như vô nghĩa khi đứng riêng, vì không còn biết nó thuộc điều nào.

In [4]:
so_tu = arts["text"].str.split().str.len().fillna(0)
dai = arts[(so_tu > 400) & (so_tu < 700)].iloc[0]
doan = chunking.split_article(dai["title"], dai["text"],
                              config.CHUNK_WORDS, config.CHUNK_OVERLAP)
print(f"{dai['article_id']}  |  {dai['title']}")
print(f"{int(so_tu.loc[dai.name])} từ  ->  {len(doan)} đoạn\n")
for i, d in enumerate(doan):
    print(f"  đoạn {i}: {d[:105]} ...")

01/2009/tt-bnn+6  |  Điều 6. Trang bị dụng cụ, sổ sách
552 từ  ->  5 đoạn

  đoạn 0: Điều 6. Trang bị dụng cụ, sổ sách. 1. Lực lượng tuần tra, canh gác đê được trang bị: - Dụng cụ thông tin, ...
  đoạn 1: Điều 6. Trang bị dụng cụ, sổ sách. 2. Số lượng dụng cụ, sổ sách tối thiểu được trang bị cho mỗi đội tuần  ...
  đoạn 2: Điều 6. Trang bị dụng cụ, sổ sách. tuần tra canh gác theo từng ca, kíp trong ngày; ghi chỉ thị, ý kiến củ ...
  đoạn 3: Điều 6. Trang bị dụng cụ, sổ sách. 3. Kinh phí mua sắm dụng cụ, sổ sách quy định tại khoản 2 của Điều này ...
  đoạn 4: Điều 6. Trang bị dụng cụ, sổ sách. 7. Việc giao nhận các dụng cụ và sổ sách trên đây phải được lập biên b ...


## 5. Ba tầng trên một câu hỏi thật

Ba tầng chạy trên cùng một câu hỏi, để thấy chúng khác nhau ở đâu.

- **BM25** khớp đúng từ. Nó bắt được số hiệu văn bản, thuật ngữ luật, con số.
- **Bi-encoder** so nghĩa. Nó bắt được cách diễn đạt khác mà không cần trùng từ.
- **Hợp nhất** chuẩn hóa min-max điểm của hai tầng rồi cộng theo trọng số
  `alpha = 0,75`. Trọng số này quét trên tập **val**, không phải test.

Chú ý: chỉ mục BM25 được dựng lại tại chỗ với tham số đã chốt (`k1 = 0,6`,
`b = 0,9`), mất khoảng ba giây. Chỉ mục vector thì nạp từ đĩa, vì mã hóa lại
154.176 đoạn mất nhiều phút.

Mô hình dùng ở đây là **AITeamVN**, mô hình sạch. Mục 7 sẽ giải thích vì sao không
dùng mô hình có điểm cao hơn.

Câu hỏi chọn làm ví dụ đúng là một trong 17 ca mà hệ bị chấm sai. Mục 9 sẽ cho
thấy vì sao nhóm cho rằng ở ca này chính **nhãn** mới đáng ngờ, chứ không phải hệ.

In [5]:
CAU_HOI = "Không đăng ký tạm trú cho khách nước ngoài phạt bao nhiêu tiền?"

ts = pipeline.doc_tham_so()
print("Tham số đã chốt trên val:", {k: ts[k] for k in ("bm25", "dense", "weighted")})

idx_bm = lexical.BM25Index(pd.read_parquet(config.ARTICLES_PATH), **ts["bm25"])
idx_de = dense.DenseIndex.load(config.INDEX_DIR / ts["dense"]["mo_hinh"])

run_bm = idx_bm.search_tokens(textnorm.tokens(CAU_HOI), config.TOPK_FUSION)
run_de = idx_de.search(idx_de.encode_queries([CAU_HOI]), top_k=config.TOPK_FUSION,
                       pooling=ts["dense"]["gop_doan"], chunk_top=config.CHUNK_TOP)[0]
run_hop = fusion.weighted_sum(run_bm, run_de, ts["weighted"]["alpha"],
                              config.TOPK_FUSION)

TIEU_DE = dict(zip(arts["article_id"], arts["title"]))

def bang_top(run, ten, k=5):
    return pd.DataFrame({
        "tầng": ten,
        "hạng": range(1, k + 1),
        "điều luật": [a for a, _ in run[:k]],
        "tiêu đề": [TIEU_DE.get(a, "")[:52] for a, _ in run[:k]],
        "điểm": [round(s, 4) for _, s in run[:k]],
    })

print("\nHỎI:", CAU_HOI)
display(pd.concat([bang_top(run_bm, "BM25"),
                   bang_top(run_de, "Ngữ nghĩa"),
                   bang_top(run_hop, "Hợp nhất")], ignore_index=True))

Tham số đã chốt trên val: {'bm25': {'k1': 0.6, 'b': 0.9}, 'dense': {'mo_hinh': 'aiteam', 'gop_doan': 'max'}, 'weighted': {'alpha': 0.75, 'recall@10': 0.9563}}


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:  90%|████████▉ | 351/391 [00:00<00:00, 3491.00it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3667.19it/s]


HỎI: Không đăng ký tạm trú cho khách nước ngoài phạt bao nhiêu tiền?


,tầng,hạng,điều luật,tiêu đề,điểm
0,BM25,1,44/2011/tt-bca+7,Điều 7. Trách nhiệm của doanh nghiệp lữ hành quốc tế,12.0676
1,BM25,2,167/2013/nđ-cp+8,Điều 8. Vi phạm quy định về đăng ký và quản lý cư tr,11.0856
2,BM25,3,167/2013/nđ-cp+17,"Điều 17. Vi phạm các quy định về xuất cảnh, nhập cản",10.4225
3,BM25,4,162/2013/nđ-cp+16,"Điều 16. Vi phạm quy định về đăng ký, đăng kiểm và g",10.3410
4,BM25,5,158/2013/nđ-cp+43,Điều 43. Vi phạm quy định về kinh doanh đại lý lữ hà,10.2883
5,Ngữ nghĩa,1,167/2013/nđ-cp+8,Điều 8. Vi phạm quy định về đăng ký và quản lý cư tr,0.6249
6,Ngữ nghĩa,2,45/2019/nđ-cp+5,Điều 5. Mức phạt tiền và thẩm quyền phạt tiền trong,0.5721
7,Ngữ nghĩa,3,167/2013/nđ-cp+17,"Điều 17. Vi phạm các quy định về xuất cảnh, nhập cản",0.5707
8,Ngữ nghĩa,4,158/2013/nđ-cp+45,Điều 45. Vi phạm quy định về kinh doanh lưu trú du l,0.5356
9,Ngữ nghĩa,5,45/2019/nđ-cp+12,Điều 12. Vi phạm quy định về kinh doanh dịch vụ lưu,0.5311


## 6. Giải thích: từ nào đã khớp

Đây là thứ tầng ngữ nghĩa **không** làm được. Nó chỉ đưa ra một con số cosine, còn
BM25 chỉ được đúng những từ đã khớp và mức đóng góp của từng từ.

Với một hệ tra cứu luật thì đây không phải chi tiết phụ. Người dùng cần kiểm tra
được vì sao hệ trả về điều này chứ không phải điều kia, nhất là khi họ sắp viện
dẫn nó.

Đóng góp tính xấp xỉ bằng `IDF(t)` nhân với phần hãm tần suất kiểu BM25. IDF cao
nghĩa là từ hiếm, mà từ hiếm mới là thứ phân biệt. Những từ như "quy_định" hay
"thực_hiện" xuất hiện ở gần như mọi điều luật nên IDF của chúng rất thấp.

In [6]:
IDF = explain.build_idf(list(pd.read_parquet(config.ARTICLES_PATH,
                                             columns=["tokens"])["tokens"]))
VAN_BAN = dict(zip(arts["article_id"], arts["text"]))

for ten, run in (("BM25", run_bm), ("Hợp nhất", run_hop)):
    aid = run[0][0]
    khop = explain.matched_terms(
        CAU_HOI, f"{TIEU_DE.get(aid, '')} {VAN_BAN.get(aid, '')}", IDF)
    print(f"[{ten}] hạng 1: {aid}  |  {TIEU_DE.get(aid, '')[:58]}")
    print(f"        từ khớp: {explain.to_chuoi(khop)}\n")

[BM25] hạng 1: 44/2011/tt-bca+7  |  Điều 7. Trách nhiệm của doanh nghiệp lữ hành quốc tế
        từ khớp: khách (7.8509), tạm_trú (7.4547), nước_ngoài (3.0745), đăng_ký (2.1246)

[Hợp nhất] hạng 1: 167/2013/nđ-cp+8  |  Điều 8. Vi phạm quy định về đăng ký và quản lý cư trú
        từ khớp: tạm_trú (10.9196), phạt (4.4709), đăng_ký (3.99), tiền (3.1078), nước_ngoài (2.236)



## 7. Kết quả trên câu có dấu

Mục này trả lời **CH1**: với câu hỏi có dấu đầy đủ, ghép BM25 vào còn giúp được bao
nhiêu.

Bảng này đọc **thẳng** từ `reports/eval/model_comparison.csv`, không con số nào gõ
tay. Chạy lại pipeline là bảng tự đổi theo.

Cột `nhiem_ban` là phần đáng chú ý nhất. Model card của
`bkai-foundation-models/vietnamese-bi-encoder` ghi rõ nó được huấn luyện trên
*"80% of the training set from the Legal Text Retrieval Zalo 2021 challenge"*. Tập
test của đồ án lại cắt ra từ đúng tập train đó, nên mô hình đã nhìn thấy phần lớn
câu hỏi test **kèm nhãn đúng**. Gọi nó là "zero-shot" là sai.

Vì vậy con số chính thức của đồ án lấy theo `AITeamVN`, mô hình có model card ghi
*"Our model was not trained on this dataset"*. Các dòng nhiễm bẩn vẫn để lại, để
thấy một con số benchmark bị thổi lên dễ thế nào.

Bảng thứ hai đo **cùng một thước**: tỷ lệ câu hỏi có ít nhất một điều gold trong
top-10. Không trộn với Recall@10 trung bình, vì câu hỏi có nhiều gold cho ra recall
lẻ và hai con số sẽ lệch nhau vài phần trăm.

In [7]:
mc = pd.read_csv(config.EVAL_DIR / "model_comparison.csv", **DOC)
display(mc[["he_thong", "recall@1", "recall@10", "mrr@10", "ndcg@10",
            "latency_p50_ms", "nhiem_ban"]])

print("Tỷ lệ câu hỏi có ít nhất một điều gold trong top-10:")
display(pd.read_csv(config.EVAL_DIR / "fusion_ceiling.csv", **DOC))

print("Đối chiếu chéo BM25 và tầng ngữ nghĩa (mô hình sạch):")
display(pd.read_csv(config.EVAL_DIR / "crossover.csv", **DOC))

,he_thong,recall@1,recall@10,mrr@10,ndcg@10,latency_p50_ms,nhiem_ban
0,BM25,0.4740,0.8522,0.6025,0.6625,0.5,NaN
1,Dense bkai,0.8204,0.9746,0.8865,0.9082,265.4,Huấn luyện trên 80% tập train Zalo 2021 (theo model card)
2,Dense aiteam,0.7811,0.9721,0.8595,0.8869,351.5,NaN
3,Dense bkai_ft,0.8141,0.9784,0.8841,0.9073,260.8,"Kế thừa nhiễm bẩn của bkai, cộng thêm fine-tune của nhóm"
4,RRF (BM25 + aiteam),0.6701,0.9670,0.7894,0.8330,352.0,NaN
5,Trọng số (BM25 + aiteam),0.7824,0.9772,0.8619,0.8900,352.0,NaN
6,RRF (BM25 + bkai_ft),0.7665,0.9860,0.8554,0.8874,261.3,"Kế thừa nhiễm bẩn của bkai, cộng thêm fine-tune của nhóm"
7,Trọng số (BM25 + bkai_ft),0.8452,0.9860,0.9070,0.9264,261.3,"Kế thừa nhiễm bẩn của bkai, cộng thêm fine-tune của nhóm"


Tỷ lệ câu hỏi có ít nhất một điều gold trong top-10:


,he_thong,ty_le_dung_top10_pct
0,BM25,85.28
1,Dense aiteam,97.34
2,Trọng số (BM25 + aiteam),97.84
3,Hợp nhất chỉ từ top-10 của hai tầng (trần của phép hợp),97.72


Đối chiếu chéo BM25 và tầng ngữ nghĩa (mô hình sạch):


,o,so_cau_hoi,ty_le_pct
0,cả hai đúng,669,84.90
1,chỉ BM25 đúng,3,0.38
2,chỉ ngữ nghĩa đúng,98,12.44
3,cả hai sai,18,2.28


## 8. Câu hỏi gõ không dấu

Mục này trả lời **CH2**. Người dân hay gõ không dấu khi nhắn trên điện thoại hoặc
gõ vội. Nhóm lấy đúng 788 câu hỏi test, bỏ hết dấu bằng máy, giữ nguyên nhãn, rồi
chạy lại toàn bộ hệ.

**Sụp đổ.** BM25 tìm theo từ, mà "phat" và "phạt" là hai từ khác nhau nên gần như
không khớp gì. Mô hình ngữ nghĩa cũng sụp; giả thuyết của nhóm là nó học chủ yếu
trên văn bản có dấu, nhưng chưa kiểm chứng trực tiếp.

**Sửa bằng phục hồi dấu.** `diacritics.PhucHoiDau` là một mô hình bigram trên âm
tiết, học từ chính kho điều luật cộng câu hỏi train:

1. Đếm unigram và bigram âm tiết.
2. Mỗi âm tiết không dấu có nhiều ứng viên: `phat` có thể là phát, phạt, phắt.
3. Viterbi chọn chuỗi ứng viên có tổng log xác suất bigram lớn nhất, với xác suất
   bigram nội suy cùng unigram theo hệ số `lambda`.

`lambda` chốt trên val theo độ chính xác âm tiết. Âm tiết người dùng đã gõ có dấu
thì giữ nguyên, nên câu gõ dấu một nửa vẫn dùng được.

**Vì sao học từ kho luật mà không tải mô hình có sẵn.** Từ vựng cần phục hồi đúng
nhất là từ vựng pháp lý, và nó có sẵn trong kho. Nhược điểm cũng từ đó mà ra: từ
nói thường như "tài xế" không có trong văn bản luật nên dễ bị phục hồi sai. Ô cuối
của mục này in ra đúng những ca đó.

**Bộ định tuyến.** `diacritics.co_dau` kiểm câu có chữ mang dấu hay không. Câu có
dấu đi thẳng qua hệ cũ, nên không có rủi ro làm hỏng câu có dấu.

Mọi bảng dưới đây đọc từ `reports/eval/khong_dau*.csv`, do
`scripts/07_khong_dau.py` sinh ra.

In [8]:
ph = diacritics.PhucHoiDau.load(config.PHUC_HOI_DAU_PATH)
VI_DU = "Đi xe máy không đội mũ bảo hiểm bị phạt bao nhiêu tiền?"
KD = diacritics.bo_dau(VI_DU)
PH = ph.phuc_hoi(KD)
print("Có dấu      :", VI_DU)
print("Không dấu   :", KD, "| có dấu?", diacritics.co_dau(KD))
print("Phục hồi dấu:", PH)

def top_ngu_nghia(cau, k=3):
    return idx_de.search(idx_de.encode_queries([cau]), top_k=k,
                         pooling=ts["dense"]["gop_doan"], chunk_top=config.CHUNK_TOP)[0]

display(pd.concat([bang_top(top_ngu_nghia(KD), "Ngữ nghĩa, không dấu", 3),
                   bang_top(top_ngu_nghia(PH), "Ngữ nghĩa, phục hồi dấu", 3)],
                  ignore_index=True))

print("Recall@10 trên 788 câu hỏi test:")
kd = pd.read_csv(config.EVAL_DIR / "khong_dau.csv", **DOC)
display(kd[["dieu_kien", "cach_xu_ly", "he_thong", "recall@1", "recall@10", "mrr@10"]])

tham = json.loads((config.EVAL_DIR / "khong_dau_params.json").read_text(encoding="utf-8"))
print(f"Phục hồi dấu: {tham['do_chinh_xac_am_tiet_test']:.2%} âm tiết đúng, "
      f"{tham['so_cau_phuc_hoi_dung_hoan_toan_test']}/{tham['so_cau_test']} câu đúng hoàn toàn, "
      f"{tham['do_tre_phuc_hoi_ms']} ms mỗi câu")
display(pd.read_csv(config.EVAL_DIR / "khong_dau_loi.csv", **DOC))

vd = pd.read_csv(config.EVAL_DIR / "khong_dau_vi_du.csv", **DOC)
sai = vd[vd["am_tiet_dung"] < vd["am_tiet"]]
display(sai[["goc", "phuc_hoi", "goc_dung_top10", "phuc_hoi_dung_top10"]].head(5))

Có dấu      : Đi xe máy không đội mũ bảo hiểm bị phạt bao nhiêu tiền?
Không dấu   : Di xe may khong doi mu bao hiem bi phat bao nhieu tien? | có dấu? False
Phục hồi dấu: Đi xe máy không đội mũ bảo hiểm bị phạt bao nhiêu tiền?


,tầng,hạng,điều luật,tiêu đề,điểm
0,"Ngữ nghĩa, không dấu",1,52/2014/tt-bca+14,"Điều 14. Bảo quản, bảo dưỡng định kỳ",0.3062
1,"Ngữ nghĩa, không dấu",2,21/2010/tt-bgtvt+3,Điều 3. Giải thích từ ngữ,0.2891
2,"Ngữ nghĩa, không dấu",3,10/2020/nđ-cp+6,Điều 6. Kinh doanh vận tải hành khách bằng xe taxi,0.2840
3,"Ngữ nghĩa, phục hồi dấu",1,100/2019/nđ-cp+6,"Điều 6. Xử phạt người điều khiển xe mô tô, xe gắn má",0.6619
4,"Ngữ nghĩa, phục hồi dấu",2,06/2013/ttlt-bkhcn-bct-bca-bgtvt+10,Điều 10. Hướng dẫn về xử lý vi phạm,0.6173
5,"Ngữ nghĩa, phục hồi dấu",3,100/2019/nđ-cp+11,Điều 11. Xử phạt các hành vi vi phạm khác về quy tắc,0.6056


Recall@10 trên 788 câu hỏi test:


,dieu_kien,cach_xu_ly,he_thong,recall@1,recall@10,mrr@10
0,có dấu,giữ nguyên,BM25,0.4740,0.8522,0.6025
1,có dấu,giữ nguyên,Ngữ nghĩa,0.7811,0.9721,0.8595
2,có dấu,giữ nguyên,Hợp nhất,0.7824,0.9772,0.8619
3,không dấu,giữ nguyên,BM25,0.0152,0.0622,0.0271
4,không dấu,giữ nguyên,Ngữ nghĩa,0.0317,0.1244,0.0582
5,không dấu,giữ nguyên,Hợp nhất,0.0343,0.1447,0.0649
6,không dấu,chỉ mục bỏ dấu,BM25 bỏ dấu,0.3230,0.7430,0.4597
7,không dấu,chỉ mục bỏ dấu,Hợp nhất BM25 bỏ dấu,0.3687,0.7557,0.4944
8,không dấu,phục hồi dấu,BM25,0.4600,0.8395,0.5881
9,không dấu,phục hồi dấu,Ngữ nghĩa,0.7659,0.9645,0.8458


Phục hồi dấu: 98.41% âm tiết đúng, 619/788 câu đúng hoàn toàn, 0.98 ms mỗi câu


,nhom,so_cau,ty_le_dung_top10_cau_goc_pct,ty_le_dung_top10_phuc_hoi_pct
0,phục hồi đúng hoàn toàn,619,98.71,98.71
1,sai ít nhất một âm tiết,169,94.67,89.94
2,mất do phục hồi sai,9,NaN,NaN
3,được nhờ phục hồi,1,NaN,NaN


,goc,phuc_hoi,goc_dung_top10,phuc_hoi_dung_top10
0,Việc sử dụng tài sản công của cơ quan để quyên góp từ thiện có t...,Việc sử dụng tài sản công của cơ quan để quyên góp từ thiện cơ t...,1,1
2,Viên chức chuyển công tác có giao hồ sơ gốc cho đơn vị mới không?,Viên chức chuyên công tác có giao hồ sơ gốc cho đơn vị mới không?,1,1
5,Chạy xe máy mà trong hơi thở hoặc máu có nồng độ cồn thì mức phạ...,Chạy xe máy mà trong hơi thở hoặc mẫu có nồng độ cồn thì mức phạ...,1,1
7,"Tài xế xe khách đe dọa, xúc phạm, tranh giành, lôi kéo hành khác...","Tải xe xe khách đe dọa, xúc phạm, tranh giành, lôi kéo hành khác...",1,1
8,Mức phạt đối với hành vi điều khiển xe máy có sử dụng ô (dù),Mức phạt đối với hành vi điều khiển xe máy có sử dụng ở (dự),1,1


## 9. Hệ sai ở đâu

Hệ sạch tốt nhất sai 17 trên 788 câu hỏi. Cả 17 ca đều được **đọc tay**, không
đoán. Nhãn nguyên nhân giữ ở tệp nguồn `docs/nhan_loi_doc_tay.csv` rồi ghép vào,
chứ không gõ thẳng vào tệp kết quả: gõ thẳng thì lần chạy sau mất sạch.

Nhóm lớn nhất là **thiếu ngữ cảnh pháp lý**: câu hỏi dùng lời nói thường, điều luật
dùng thuật ngữ, và không tầng nào bắc được cầu giữa hai bên.

Nhóm cuối là **nhãn gold đáng ngờ**: có hai ca mà điều luật hệ trả về sát câu hỏi
hơn cả điều được ghi trong nhãn. Nhóm không giấu chỗ này, vì nó củng cố nhận định
ở mục 3 rằng bộ nhãn không đầy đủ.

In [9]:
display(pd.read_csv(config.EVAL_DIR / "error_summary.csv", **DOC))

sai = pd.read_csv(config.EVAL_DIR / "error_taxonomy.csv", **DOC)
for _, h in sai[sai["nguyen_nhan"] == "sai_gold"].head(2).iterrows():
    print("HỎI      :", h["cau_hoi"])
    print("Nhãn ghi :", h["gold"], "|", str(h["gold_title"])[:62])
    print("Hệ trả về:", h["top1"], "|", str(h["top1_title"])[:62])
    print("Nhận xét :", h["ghi_chu"], "\n")

,nguyen_nhan,so_ca,ty_le_pct
0,thieu_ngu_canh,10,58.8
1,nhieu_dieu_dung,5,29.4
2,sai_gold,2,11.8


HỎI      : Không đăng ký tạm trú cho khách nước ngoài phạt bao nhiêu tiền?
Nhãn ghi : 47/2014/qh13+33 | Điều 33. Khai báo tạm trú
Hệ trả về: 167/2013/nđ-cp+8 | Điều 8. Vi phạm quy định về đăng ký và quản lý cư trú
Nhận xét : Câu hỏi hỏi mức phạt, gold lại là điều quy định nghĩa vụ khai báo; điều hệ trả về đúng hơn 

HỎI      : Người chấp hành biện pháp bắt buộc chữa bệnh sẽ chữa bệnh ở đâu?
Nhãn ghi : 41/2019/qh14+133 | Điều 133. Cơ quan, tổ chức được giao nhiệm vụ thi hành biện ph
Hệ trả về: 41/2019/qh14+138 | Điều 138. Tổ chức điều trị cho người bị bắt buộc chữa bệnh
Nhận xét : Hệ trả Điều 138 'Tổ chức điều trị cho người bị bắt buộc chữa bệnh', sát câu hỏi hơn gold 



## 10. Trợ lý tra cứu

Đây là một chatbot kiểu **truy hồi**, không phải kiểu RAG. Nó không có mô hình sinh
chữ, nên không bịa được câu nào: câu trả lời là đúng khoản luật có thật trong kho,
in nguyên văn, kèm số hiệu điều để người dùng tự kiểm.

`tra_cuu.TroLyTraCuu` ghép lại toàn bộ đường chạy:

1. Kiểm dấu. Câu không dấu thì phục hồi dấu trước.
2. BM25 và tầng ngữ nghĩa, hợp nhất bằng tổng có trọng số đã chốt trên val.
3. Với điều luật đứng đầu, tìm **đoạn** có cosine cao nhất với câu hỏi. Đó chính
   là khoản quyết định điểm `max` khi gộp đoạn về điều, nên nó là phần trả lời.
4. In kèm những từ đã khớp và hai điều luật tham khảo thêm.

Trợ lý dùng lại chỉ mục đã nạp ở mục 5, không nạp mô hình lần hai.

Vì sao không dùng LLM sinh câu trả lời: trả sai điều luật kèm một câu trả lời trôi
chảy còn nguy hiểm hơn trả sai điều luật, vì người đọc không còn thấy chỗ sai.

In [10]:
tro_ly = tra_cuu.TroLyTraCuu(bm=idx_bm, de=idx_de)
print(tro_ly.tra_loi("di xe may khong doi mu bao hiem bi phat bao nhieu tien"))

Bạn hỏi: di xe may khong doi mu bao hiem bi phat bao nhieu tien
(Câu không dấu, đã phục hồi thành: đi xe máy không đội mũ bảo hiểm bị phạt bao nhiêu tiền)

Điều luật phù hợp nhất: Điều 6. Xử phạt người điều khiển xe mô tô, xe gắn máy (kể cả xe máy điện), các loại xe tương tự xe mô tô và các loại xe tương tự xe gắn máy vi phạm quy tắc giao thông đường bộ  [100/2019/nđ-cp+6]
Nội dung liên quan: “đỗ xe tại nơi đường bộ giao nhau cùng mức với đường sắt; dừng xe, đỗ xe trong phạm vi an toàn của đường sắt, trừ hành vi vi phạm quy định tại điểm b khoản 2, điểm b khoản 3 Điều 49 Nghị định này; i) Không đội “mũ bảo hiểm cho người đi mô tô, xe máy” hoặc đội “mũ bảo hiểm cho người đi mô tô, xe máy” không cài quai đúng quy cách khi điều khiển xe tham gia giao thông trên đường bộ; k) Chở người ngồi trên xe không đội “mũ bảo hiểm cho người đi mô tô, xe máy” hoặc đội “mũ bảo hiểm cho người đi mô tô, xe máy” không cài quai đúng quy cách, trừ trường hợp chở người bệnh đi cấp cứu, trẻ em dưới 06 ...”
Từ

### Tự gõ câu hỏi

Đổi câu trong ô dưới rồi chạy lại. Có dấu hay không dấu đều được.

Nếu trợ lý trả sai thì **đừng giấu**: 17 ca sai của câu có dấu đã được phân loại ở
mục 9, và một ca sai mới chỉ xác nhận đúng những gì đã báo cáo. Với câu không dấu,
xem dòng "đã phục hồi thành" trước: nếu phục hồi sai một từ nói thường thì đó là
đúng loại lỗi đã nêu ở mục 8.

In [11]:
print(tro_ly.tra_loi("Nguoi lao dong nghi viec co duoc tra luong nhung ngay chua nghi phep khong?"))

Bạn hỏi: Nguoi lao dong nghi viec co duoc tra luong nhung ngay chua nghi phep khong?
(Câu không dấu, đã phục hồi thành: Người lao động nghỉ việc có được trả lương những ngày chưa nghỉ phép không?)

Điều luật phù hợp nhất: Điều 18. Tiền lương tính trả cho ngày nghỉ hằng năm, nghỉ lễ, tết  [19/2014/tt-blđtbxh+18]
Nội dung liên quan: “3. Người lao động do chấm dứt hợp đồng lao động hoặc vì lý do khác mà chưa nghỉ hàng năm hoặc chưa nghỉ hết số ngày nghỉ hàng năm theo quy định thì được người sử dụng lao động thanh toán tiền lương những ngày người lao động chưa nghỉ. Tiền lương làm căn cứ tính trả cho những ngày người lao động chưa nghỉ là tiền lương tháng ghi trên hợp đồng lao động bình quân 6 tháng trước khi chấm dứt hợp đồng lao động hoặc trước khi tính trả cho người lao động, chia cho số ngày làm việc bình thường trong tháng theo quy định của pháp luật mà hai bên xác định nhưng tối đa không quá 26 ngày, nhân với số ngày ...”
Từ khớp: trả (5.0992), lao_động (4.9747), ngày (2.5386), người

## Kết luận

| | Recall@10 trên test |
|---|---|
| CH1. Câu có dấu, hệ lai | 0,9772, hơn ngữ nghĩa thuần 0,50 điểm (4 câu trên 788) |
| CH2. Câu không dấu, giữ nguyên hệ | 0,1447 |
| Câu không dấu, BM25 chỉ mục bỏ dấu | 0,7430 |
| Câu không dấu, phục hồi dấu rồi hệ lai | 0,9670, phục hồi đúng 98,41% âm tiết |

**CH1.** Với câu có dấu, hợp nhất có giúp nhưng rất ít. Không gọi 0,50 điểm phần
trăm là đáng kể, nhất là khi mỗi cấu hình chỉ chạy một seed nên nhóm không có cơ
sở nói về biên độ nhiễu.

**CH2.** Với câu không dấu, cả hệ sụp. BM25 trên chỉ mục bỏ dấu là cách rẻ nhất, không
cần mô hình nào, và lúc này BM25 lại là tầng đứng vững nhất. Cách tốt nhất là phục
hồi dấu bằng bigram học từ kho luật: hệ về lại gần mức câu có dấu, kém 1,02 điểm.
Chỗ phục hồi sai tập trung ở từ nói thường mà văn bản luật không dùng.

Giới hạn cần nói rõ: câu không dấu ở đây tạo bằng máy từ câu gốc, chưa có câu do
người thật gõ, và chưa đo trên câu gõ dấu một nửa hay sai chính tả.

Bài học về dữ liệu: chống rò rỉ không dừng ở việc chia lại tập. Mô hình bi-encoder
tiếng Việt phổ biến nhất đã được huấn luyện trên chính bộ dữ liệu này.